# Coleta de notícias sobre Juizados Especiais

Usei Beautiful Soup para retirar as dez notícias mais recentes da página dos Juizados Especiais no portal do CNJ. As notícias complementam o indicador estatístico com informações de contexto.

In [ ]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
#feito com auxilio de IA chat GPT

PASTA_PROJETO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()), None)
if PASTA_PROJETO is None:
    raise FileNotFoundError("Não encontrei a pasta data. Abra o notebook dentro do projeto TP2.")
ARQUIVO_BRUTO = PASTA_PROJETO / "data" / "raw" / "noticias_juizados.html"
ARQUIVO_TRATADO = PASTA_PROJETO / "data" / "processed" / "noticias_juizados.csv"

In [ ]:
resposta = requests.get(
    "https://www.cnj.jus.br/programas-e-acoes/juizados-especiais/noticias/",
    headers={"User-Agent": "Mozilla/5.0 (projeto acadêmico TP2)"},
    timeout=120,
)
print("Status da página:", resposta.status_code)
resposta.raise_for_status()
ARQUIVO_BRUTO.write_text(resposta.text, encoding="utf-8")

In [ ]:
artigos = BeautifulSoup(resposta.text, "html.parser").select("article.elementor-post")[:10]
print("Notícias encontradas:", len(artigos))
if not artigos:
    raise RuntimeError("Nenhuma notícia encontrada. Verifique o seletor da página do CNJ.")
print(artigos[0].select_one(".elementor-post__title a").get_text(" ", strip=True))

In [ ]:
linhas = []

for artigo in artigos:
    titulo = artigo.select_one(".elementor-post__title a")
    linhas.append({
        "titulo": titulo.get_text(" ", strip=True),
        "data": artigo.select_one(".elementor-post-date").get_text(" ", strip=True),
        "resumo": artigo.select_one(".elementor-post__excerpt").get_text(" ", strip=True),
        "url": titulo["href"],
        "fonte": "Conselho Nacional de Justiça",
    })

noticias = pd.DataFrame(linhas)
noticias

In [ ]:
assert len(noticias) == 10
assert noticias["titulo"].notna().all()
assert noticias["url"].nunique() == 10

print("Testes concluídos sem erros.")

In [ ]:
noticias.to_csv(ARQUIVO_TRATADO, index=False, encoding="utf-8-sig")
print("Arquivo bruto:", ARQUIVO_BRUTO)
print("Arquivo tratado:", ARQUIVO_TRATADO)